# Chapter 4: Pretraining and loss evaluation

In the last chapter we assembled a complete GPT model: embeddings, a stack of transformer blocks, and an output layer that scores every word in the vocabulary. When we asked it to write, it produced random tokens. That was expected. Every weight was still a random number, so the model had nothing to say.

This chapter is about **teaching it**. We will not start by throwing a huge dataset at the 124-million-parameter network. We will first make the training machinery so small and so visible that every piece can be checked by hand, then scale that same machinery up to a real GPT, and finally load OpenAI's public GPT-2 weights into the classes we wrote ourselves.

Here is the path we will follow:

- **Logits and targets**: the two tensors a language model is judged on
- **Cross-entropy loss**: a single number that says how surprised the model is by the correct next token
- **One training epoch**: forward pass, loss, backward pass, optimizer step
- **The GPT model from Chapter 3**: reused here as the thing we actually train
- **A tiny dummy GPT**: proof that the wiring works, without waiting on 162 million parameters
- **Validation loss**: scoring the model without accidentally updating it
- **The master training loop**: many epochs, train vs. validation, tokens seen
- **Checkpoints**: saving and loading `state_dict`s so work is not thrown away
- **OpenAI GPT-2 weights**: downloading the public 124M checkpoint and translating it into our layer names

A quick reminder about shapes, because they will keep us honest:

- **logits** have shape `(batch, tokens, vocab_size)`: at every position, one raw score per word in the vocabulary
- **targets** have shape `(batch, tokens)`: at every position, the correct next-token ID
- **loss** is a single number, the average cross-entropy over those positions

## Step 1: Logits, targets, and what "wrong" looks like

To train a model we need a number that says how wrong it currently is. For a GPT, "wrong" has a precise meaning: **at each position, did it assign a high score to the token that actually comes next?**

The cell below builds the smallest possible example of that setup, with no model involved yet. We pretend the vocabulary has only **5** tokens, and we have **2** sequences of **3** tokens each.

- `inputs_logits` has shape `(2, 3, 5)`. Each of the 3 positions in each of the 2 sequences holds 5 scores, one per vocabulary item.
- `targets` has shape `(2, 3)`. Each entry is the correct token ID at that position:
  - first sequence: tokens `0`, `1`, `2`
  - second sequence: tokens `3`, `4`, `0`

The logits are not random. They are stacked so the **largest score often sits on the correct target**. For example, the first token of the first sequence has scores `[2.0, 1.0, 0.1, 0.5, 0.2]`, and the target is `0`, which matches the `2.0`. That means the loss we compute next should come out reasonably small. If it does, we know we are comparing the right positions to the right labels.

Print the two shapes before moving on. Everything else in this chapter is the same idea, just with bigger numbers.

In [50]:
import torch 
import torch.nn as nn 
inputs_logits = torch.tensor([
    [[2.0, 1.0, 0.1, 0.5, 0.2],
     [0.5, 2.5, 0.3, 0.8, 0.1],
     [0.1, 0.4, 3.0, 0.2, 0.5]],

     [[1.5, 0.3, 0.5, 2.0, 0.2],
      [0.2, 0.8, 0.1, 0.4, 2.2],
      [3.5, 0.1, 0.2, 0.5, 0.4]]
])

targets= torch.tensor([[0,1,2],
[3,4,0]])

print("logits shape:", inputs_logits.shape)
print("targets shape:", targets.shape)

logits shape: torch.Size([2, 3, 5])
targets shape: torch.Size([2, 3])


## Step 2: Flattening tensors and computing cross-entropy

PyTorch's `CrossEntropyLoss` does not want a 3-dimensional batch of sequences. It wants a simpler view of the same data:

- **predictions** of shape `(N, C)`: `N` separate examples, `C` class scores each
- **targets** of shape `(N,)`: one correct class index per example

Our GPT-style tensors are sequence-shaped, so we reshape them without changing any values:

- `inputs_logits.flatten(0, 1)` merges the batch and token dimensions. Shape goes from `(2, 3, 5)` to `(6, 5)`: 6 next-token questions, 5 scores each.
- `targets.flatten()` does the same for the labels, from `(2, 3)` to `(6,)`.

Every token position is now treated as its own classification problem. That is exactly how a language model is trained: **at every position, predict the next token**.

### What cross-entropy actually measures

For one position, the loss is:

$$\mathcal{L} = -\log \, \text{softmax}(z)_{y}$$

where $z$ is the vector of 5 logits and $y$ is the correct token ID. Softmax turns the logits into probabilities that sum to 1. We then look up the probability of the correct token and take `-log` of it.

- If the model puts almost all of its probability on the correct token, `-log` is close to **0** (not surprised).
- If it puts almost no probability on the correct token, `-log` is a **large** number (very surprised).

`nn.CrossEntropyLoss` does this for all 6 positions and returns the average. You should see a loss around **0.46**. That is fairly small, which makes sense: we stacked the toy logits so the largest score at each position is often the correct target. A random model with a vocabulary of 5 would land closer to `-log(1/5) ≈ 1.61`.

In [51]:
flat_logits= inputs_logits.flatten(0,1)
flat_targets= targets.flatten()
loss_fn=nn.CrossEntropyLoss()
loss= loss_fn(flat_logits, flat_targets)

print("flattend logits shape:", flat_logits.shape)
print("flattened targets shape:", flat_targets.shape)
print("calculated training loss:", loss.item())

flattend logits shape: torch.Size([6, 5])
flattened targets shape: torch.Size([6])
calculated training loss: 0.45796191692352295


## Step 3: The anatomy of one training epoch

A loss on its own is just a diagnosis. Training is what we do with that diagnosis: compute how each weight contributed to the error, then nudge every weight a tiny step in the direction that would have made the error smaller.

`train_model_one_epoch` is that recipe, written as a function so we can reuse it. One **epoch** means one full pass over every batch in a dataloader.

### The loop, line by line

1. **`model.train()`** puts the model in training mode. Dropout (and similar layers) behave differently during training and evaluation, so this switch matters.
2. **`inputs.to(device), targets.to(device)`** moves the batch onto the CPU or GPU the model is sitting on. If the data and the model live on different devices, PyTorch will refuse to run.
3. **`optimizer.zero_grad()`** erases gradients left over from the previous batch. Without this, new gradients would be *added* to old ones, and the updates would be nonsense.
4. **`logits = model(inputs)`** is the forward pass. For a GPT, `inputs` are token IDs of shape `(batch, tokens)`, and `logits` come out as `(batch, tokens, vocab_size)`.
5. Flatten and compute **cross-entropy**, exactly as in Step 2. Training must use the same loss we just defined by hand, otherwise we would be optimising a different problem.
6. **`loss.backward()`** asks autograd to walk backwards through every operation that produced the loss, and to store a gradient on every learnable parameter.
7. **`optimizer.step()`** uses those gradients to update the weights. The size of each nudge is controlled by the learning rate we give the optimizer later.
8. **`total_loss += loss.item()`** accumulates a running sum so we can report how the epoch went. `.item()` turns a one-element tensor into a plain Python number, which is what we want for logging.

This function is the inner engine. The next steps attach it to a real GPT, add a validation check, and wrap it in a multi-epoch loop.

In [ ]:
import torch
def train_model_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss=0

    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs,targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()

        logits=model(inputs)

        flat_logits = logits.flatten(0,1)
        flat_targets= targets.flatten()

        loss= torch.nn.functional.cross_entropy(flat_logits, flat_targets)

        loss.backward()

        optimizer.step()
        total_loss+=loss.item()
    return total_loss 

## Step 4: Bringing in the GPT model from Chapter 3

The training helpers above are model-agnostic: they only need something that maps token IDs to logits. The something we actually want to train is the `GPTMODEL` class from the previous chapter.

`%run` is a Jupyter magic command. It executes another notebook from top to bottom, as if every cell of that notebook had been pasted into this one. When it finishes, all of Chapter 3's classes are available here:

- `LayerNormFromScratch`, `GELUActivation`, `FeedForward`
- `TransformerBlock`
- `GPTMODEL`
- `generate_text_simple`
- and the attention classes that Chapter 3 itself pulled in from Chapter 2

The path `../03_transformers/transformers.ipynb` means "go up one folder, then into `03_transformers`".

Because the whole notebook runs, you will see Chapter 3's printed shape checks appear below this cell. That is expected. Scroll past them; what we need is the classes they leave behind in memory.

In [53]:
%run ../03_transformers/transformers.ipynb

input tensor shape: torch.Size([2, 3, 5])
output tensor shape: torch.Size([2, 3, 5])

 normalised output mean (should be close to 0):
 tensor([[[-1.1921e-07],
         [-2.3842e-08],
         [ 2.3842e-08]],

        [[-1.1921e-08],
         [ 1.7881e-08],
         [ 5.9605e-09]]], grad_fn=<MeanBackward1>)

 normalised output var(should be close to 1):
 tensor([[[1.0000],
         [1.0000],
         [1.0000]],

        [[1.0000],
         [1.0000],
         [1.0000]]], grad_fn=<VarBackward0>)
input batch shape: torch.Size([8, 4, 256])
ffn output shape: torch.Size([8, 4, 256])
Note: you may need to restart the kernel to use updated packages.
inputs shape: torch.Size([3, 3])
query: tensor([0.4400, 0.1500, 0.8900])
raw attention scores: tensor([1.0082, 0.9599, 0.4495])
attention weights(percentages): tensor([0.3961, 0.3774, 0.2265])
final context vector: tensor([0.5070, 0.5803, 0.6197])
all raw scores matrix:
 tensor([[0.9995, 0.9544, 0.4438],
        [0.9544, 1.4950, 1.1058],
        [0.

## Step 5: A tiny GPT to prove the training wiring works

The real GPT-2 sized model from Chapter 3 has about 162 million parameters. That is far too heavy for a "does the wiring work?" test. Instead we build a **toy GPT** whose config is small enough to run in a blink, but still uses the exact same `GPTMODEL` class.

`GPT_CONFIG_DUMMY` shrinks every important number:

| Setting | Dummy value | Why it is small |
|---|---|---|
| `vocab_size` | 5 | A fake vocabulary of 5 tokens, matching the toy logits from Step 1 |
| `context_length` | 4 | Sequences of only 4 tokens |
| `emb_dim` | 16 | Each token is a 16-dimensional vector instead of 768 |
| `n_heads` | 2 | Two attention heads |
| `n_layers` | 2 | Two transformer blocks |
| `drop_rate` | 0.0 | No dropout, so this one-shot test is deterministic |
| `qkv_bias` | False | Matches the default attention layers we wrote |

`torch.manual_seed(123)` freezes the random starting weights, so re-running the cell gives the same loss.

### The optimizer: AdamW

**AdamW** is the optimiser almost every GPT-style model uses. It is a variant of Adam that handles **weight decay** (a small penalty on large weights) more cleanly. `lr=5e-4` is a typical starting learning rate for this kind of run: large enough that the loss moves, small enough that a single step does not explode.

### One mock training step

We do not even need a real dataloader here. Two random batches of token IDs are enough:

1. `mock_inputs` and `mock_targets` are shape `(2, 4)`: 2 sequences, 4 tokens, IDs drawn from `{0, 1, 2, 3, 4}`.
2. `optimizer.zero_grad()` clears any leftover gradients.
3. `dummy_gpt(mock_inputs)` produces logits of shape `(2, 4, 5)`.
4. Flatten and compute cross-entropy, exactly as in Step 2.
5. `loss.backward()` fills in a gradient for every weight.
6. `optimizer.step()` applies those gradients.

If the printout shows a finite loss and says the gradients were computed, the whole chain is alive: embeddings, transformer blocks, output head, loss, autograd, and AdamW. That is the green light we need before writing the full training loop.

In [54]:
import torch 
from torch.optim import AdamW

GPT_CONFIG_DUMMY= {"vocab_size": 5, "context_length": 4, "emb_dim": 16, "n_heads": 2, "n_layers": 2, "drop_rate": 0.0, "qkv_bias": False}
torch.manual_seed(123)
dummy_gpt= GPTMODEL(GPT_CONFIG_DUMMY)
optimizer= AdamW(dummy_gpt.parameters(), lr=5e-4)

mock_inputs= torch.randint(0,5, (2,4))
mock_targets = torch.randint(0,5, (2,4))

optimizer.zero_grad()
mock_logits= dummy_gpt(mock_inputs)
loss = torch.nn.functional.cross_entropy(mock_logits.flatten(0,1), mock_targets.flatten())
loss.backward()
optimizer.step()

print("mock execution loss score:", loss.item())
print("weights gradients status : successfully computed")

mock execution loss score: 1.724005103111267
weights gradients status : successfully computed


## Step 6: Guarding the validation split

If a model's training loss keeps dropping, that only proves one thing: it is getting better at the examples it is **allowed to practise on**. It might be memorising those examples instead of learning language. The only honest check is a **validation split**: a set of sequences the model is never trained on.

A healthy training run looks like this:

- **train loss and validation loss both fall** — the model is actually learning
- **train loss falls, validation loss rises** — the model is overfitting, memorising the training set

`calc_loss_loader` is the function that computes that honest number. It walks through a dataloader, measures cross-entropy, and returns the **average loss**. A few details make it safe to call in the middle of training without accidentally teaching the model extra things:

- **Empty loader guard**: if the dataloader has no batches, the function returns `nan` instead of dividing by zero.
- **`model.eval()`**: switches the model into evaluation mode. Dropout is turned off, so the same input always produces the same output. That is what we want when we are measuring, not exploring.
- **`torch.no_grad()`**: tells PyTorch not to build a computation graph. We are only scoring the model, not updating it, so there is no reason to store gradients. This saves a lot of memory.
- **Same loss recipe as training**: flatten logits from `(batch, tokens, vocab)` to `(batch * tokens, vocab)`, flatten targets, then `cross_entropy`. Training and evaluation must use the identical loss, otherwise the two numbers cannot be compared.
- **`num_batches`**: during a long training run we often do not want to wait for a full pass over the data just to print a progress line. Passing `num_batches=5` (for example) scores only the first 5 batches, which is a fast and good-enough snapshot.
- **Average, not sum**: the return value divides the total by however many batches were actually used, so the number stays on the same scale no matter how many batches you score.

The word *guarding* here means two things at once: we keep the validation data out of the training updates, and we keep evaluation itself from changing any weights.

In [55]:
import torch
def calc_loss_loader(dataloader, model, device, num_batches=None):
    total_loss=0.0
    if len(dataloader)==0:
        return float("nan")

    model.eval()

    for i,(inputs, targets), in enumerate(dataloader):
        if num_batches and i>=num_batches:
            break
        inputs, targets= inputs.to(device), targets.to(device)

        with torch.no_grad():
            logits=model(inputs)

        flat_logits = logits.flatten(0,1)
        flat_targets= targets.flatten()

        loss= torch.nn.functional.cross_entropy(flat_logits, flat_targets)
        total_loss+=loss.item()

    return total_loss/min(len(dataloader), num_batches if num_batches else len(dataloader))

## Step 7: Testing the evaluator function

Before trusting `calc_loss_loader` inside a long training run, we try it on a tiny fake dataset.

`TensorDataset` wraps two tensors into a dataset of `(input, target)` pairs. We create **4** random sequences, each **4** tokens long, with token IDs between 0 and 4 (matching the dummy GPT's vocabulary of size 5). `DataLoader` then serves those 4 sequences in batches of 2, so the loader contains exactly **2 batches**.

We score both batches (`num_batches=2`) with the dummy GPT from the previous step, on the CPU. The printed number is the average cross-entropy over those batches. It will not be a beautiful small loss, because the data is random and the model has barely been trained. That is fine. We only need to confirm that the function runs, returns a single finite number, and does not crash on a real dataloader.

This number is our **baseline**: a starting loss we can compare against later, once a real training loop has had a chance to work.

In [56]:
import torch
from torch.utils.data import TensorDataset, DataLoader

mock_inputs= torch.randint(0,5,(4,4))
mock_targets = torch.randint(0,5,(4,4))
mock_ds = TensorDataset(mock_inputs, mock_targets)
mock_loader= DataLoader(mock_ds, batch_size=2)

baseline_loss= calc_loss_loader(mock_loader, dummy_gpt, device="cpu", num_batches=2)
print("baseline loss scrore vector:", baseline_loss)

baseline loss scrore vector: 1.8556649684906006


## Step 8: Implementing the master training loop

One epoch is useful as a building block. Real pretraining repeats that block many times, periodically checks the validation loss, and keeps a history we can plot. `train_model` is that outer loop.

### What the arguments mean

| Argument | Role |
|---|---|
| `model` | The GPT model we are training |
| `train_loader` | Batches of `(input, target)` token IDs for learning |
| `val_loader` | Held-out batches, used only for scoring |
| `optimizer` | The algorithm that updates the weights (AdamW in this notebook) |
| `device` | Where tensors should live (`"cpu"` or `"cuda"`) |
| `num_epochs` | How many full passes over the training data to run |
| `eval_freq` | Evaluate every this many epochs (for example, `1` means every epoch) |
| `eval_iter` | How many batches `calc_loss_loader` should score at each evaluation |
| `start_context`, `tokenizer` | Reserved for printing a sample of generated text during training. The loop below focuses on tracking loss; generation can be plugged in later using `generate_text_simple` from Chapter 3 |

### What happens in one epoch

1. `model.train()` turns dropout back on, because we are learning again.
2. For every training batch:
   - `optimizer.zero_grad()` clears gradients from the previous batch, so they do not pile up
   - `logits = model(input_batch)` is the forward pass
   - `cross_entropy` on the flattened logits and targets is the same loss we derived in Step 2
   - `loss.backward()` walks the computation graph backwards and fills in a gradient for every weight
   - `optimizer.step()` nudges every weight a tiny step in the direction that reduces the loss
   - `tokens_seen` grows by `input_batch.numel()`, the number of token IDs in the batch. This is a better x-axis for a loss curve than "epoch number", because it measures how much text the model has actually seen.
3. When `epoch % eval_freq == 0`, we freeze the model with `calc_loss_loader` and record three numbers: training loss, validation loss, and tokens seen so far. A print line lets us watch both losses as they move.

The function returns those three lists. Plotting `train_losses` and `val_losses` against `track_tokens_seen` is the standard way to judge whether pretraining is healthy.

A useful mental picture: **training** is the model practising on homework it can look at. **Validation** is a test it has never seen. We only trust the test.

In [57]:
def train_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):

    train_losses,val_losses,track_tokens_seen= [], [], []
    tokens_seen=0

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            logits= model(input_batch)

            loss= torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
            loss.backward()
            optimizer.step()

            tokens_seen+=input_batch.numel()

        if epoch%eval_freq==0:
            train_loss= calc_loss_loader(train_loader, model, device, eval_iter)
            val_loss= calc_loss_loader(val_loader, model, device, eval_iter)

            train_losses.append(train_loss)
            val_losses.append(val_loss)
            track_tokens_seen.append(tokens_seen)

            print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss:{val_loss:.4f}")
    return train_losses, val_losses, track_tokens_seen

## Step 9: Saving and loading model weights

Pretraining can take minutes, hours, or days. We do not want to throw that work away every time the notebook restarts. A **checkpoint** is a snapshot of the model's learned numbers, written to disk so we can come back later.

PyTorch splits a model into two pieces:

- the **architecture**: the Python class, the layers, the shapes (`GPTMODEL` plus a config)
- the **state**: the actual weight tensors inside those layers

`model.state_dict()` returns the state as a dictionary: layer name → tensor. That is what we save.

The cell below does a round trip, which is the standard sanity check:

1. **`torch.save(dummy_gpt.state_dict(), "model_checkpoint.pt")`** writes the dummy GPT's weights to a file in this folder.
2. **`fresh_model = GPTMODEL(GPT_CONFIG_DUMMY)`** builds a brand new model with the **same config**. The architecture must match, otherwise the names and shapes in the file will not line up with the new model.
3. **`fresh_model.load_state_dict(torch.load(...))`** copies every saved tensor into the matching layer of `fresh_model`.

After this, `fresh_model` has the same knowledge as `dummy_gpt`, even though it is a different Python object. The original model could be deleted; the file is enough to recreate it.

Two details worth remembering for later:

- Saving the `state_dict` is usually better than saving the whole model object, because it stays compatible if you slightly change your Python code.
- When you train for real, you often also save the optimizer's `state_dict` (AdamW keeps running averages). That lets you **resume** training mid-run, not just reload a finished model.

In [58]:
import torch

torch.save(dummy_gpt.state_dict(), "model_checkpoint.pt")
print("model weights status: successfully saved to model_checkpoint.pt")

fresh_model= GPTMODEL(GPT_CONFIG_DUMMY)
fresh_model.load_state_dict(torch.load("model_checkpoint.pt"))
print("model weight status: successfully loaded info Fresh Architecture")

model weights status: successfully saved to model_checkpoint.pt
model weight status: successfully loaded info Fresh Architecture


## Step 10: Connecting with OpenAI's raw GPT-2 weights

Our dummy GPT proved that training *can* work, but a 2-layer, 16-dimensional model on random tokens will never speak English. Training a 124-million-parameter GPT from scratch needs a huge dataset and a lot of compute.

There is a shortcut that is still honest, because we built the architecture ourselves: **load the official GPT-2 weights** into our classes. OpenAI released those weights; Hugging Face's `transformers` library can download them with one line.

`GPT2Model.from_pretrained("gpt2")` fetches the smallest GPT-2 checkpoint, often called GPT-2 124M: 12 layers, 12 heads, 768-dimensional embeddings, a 50,257-token vocabulary. `state_dict()` then gives us a dictionary of **148** named tensors.

We cannot load that dictionary into our model directly. The numbers are the right numbers, but the **names and the layouts are different**. Hugging Face's first key is `wte.weight` ("word token embeddings"). Our matching tensor is called `tok_emb.weight`. Until we translate those names, PyTorch has no idea they belong together.

The print below is there to make that mismatch visible. It is the reason the next step exists: a dedicated mapping function that copies every OpenAI tensor into the matching place in our `GPTMODEL`.

In [59]:
#%pip install transformers
import transformers
import torch
from transformers import GPT2Model

openai_gpt= GPT2Model.from_pretrained("gpt2")
openai_params= openai_gpt.state_dict()
print("openai GPT2 layers found:", len(openai_params.keys()))

from torch.optim import AdamW
print("\n--- Sample Keys Mismatch Comparision ---")
print("openai layer name:", list(openai_params.keys())[0])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6305.62it/s]

openai GPT2 layers found: 148

--- Sample Keys Mismatch Comparision ---
openai layer name: wte.weight


## Step 11: The weight translation function

`assign_weights` is a dictionary-to-dictionary copy, with a few layout conversions along the way. `.copy_()` writes values **in place** into our model's existing parameter tensors, so the Python objects stay the same and only the numbers change.

### Three kinds of mismatch we have to fix

1. **Names.** OpenAI uses `wte`, `wpe`, `h.{l}`, `ln_f`. We use `tok_emb`, `pos_emb`, `trf_blocks.{l}`, `final_ln`.
2. **Fused attention.** OpenAI stores query, key and value in **one** matrix, `c_attn`, of shape `(emb_dim, 3 * emb_dim)`. We store a separate `W_query`, `W_key` and `W_value` on **each head**. So we split the big matrix into three, transpose, then split again by head.
3. **Weight layout.** Hugging Face implements those layers as `Conv1D`, which stores weights as `(in_features, out_features)`. `nn.Linear` stores them as `(out_features, in_features)`. Every linear-layer copy therefore uses `.T` (transpose) so the shapes agree.

### Walking through the copies

- **`n_layers`** and **`n_heads`** are read off our model, so the mapping stays correct if you ever build a smaller or larger GPT.
- **Token and position embeddings** are a straight copy: `wte.weight` → `tok_emb.weight`, `wpe.weight` → `pos_emb.weight`.
- **For each transformer block `l`:**
  - `torch.chunk(..., 3, dim=-1)` splits `c_attn` into Q, K, V.
  - `.T` flips each piece into `nn.Linear` layout.
  - `torch.chunk(..., n_heads, dim=0)` then slices those into one tensor per head, which lands on `trf_blocks.{l}.att.heads.{h}.W_query` (and key, and value).
  - If our model was built with `qkv_bias=True`, the same split is applied to `c_attn.bias`.
  - Attention's output projection comes from `c_proj`.
  - The feed-forward network has two linear layers with GELU in the middle. `ff.layers.0` is the expand layer (`c_fc`), `ff.layers.2` is the shrink layer (`c_proj`). Index `1` is GELU, which has no weights, so it is skipped.
  - `ln1` / `ln2` `scale` and `shift` match OpenAI's `ln_1` / `ln_2` `weight` and `bias`.
- **Final LayerNorm** comes from `ln_f`.
- **Output head.** The last line copies `wte.weight` into `out_head.weight`. That is **weight tying**: GPT-2 reuses the token embedding matrix as the classifier that scores the next token, instead of learning a second giant matrix. Chapter 3 counted about 38 million extra parameters when we did *not* tie them. Loading GPT-2 correctly means tying them here.

After this function returns, our `GPTMODEL` holds the same knowledge as the public GPT-2 124M checkpoint, just stored in the classes we wrote ourselves.

In [60]:
def assign_weights(gpt_model, openai_params):
    # 1. Access models state dict pointers
    d = gpt_model.state_dict()
    n_layers = len(gpt_model.trf_blocks)
    n_heads = len(gpt_model.trf_blocks[0].att.heads)
    
    # Global mapped lookups configuration
    d["tok_emb.weight"].copy_(openai_params["wte.weight"])
    d["pos_emb.weight"].copy_(openai_params["wpe.weight"])
    
    # 2. Iterate through Transformer Blocks layers
    for l in range(n_layers):
        # OpenAI stores Q, K, V fused in one Conv1D matrix; split and map onto each head
        q_w, k_w, v_w = torch.chunk(openai_params[f"h.{l}.attn.c_attn.weight"], 3, dim=-1)
        q_w, k_w, v_w = q_w.T, k_w.T, v_w.T
        
        for h, (q_h, k_h, v_h) in enumerate(zip(
            torch.chunk(q_w, n_heads, dim=0),
            torch.chunk(k_w, n_heads, dim=0),
            torch.chunk(v_w, n_heads, dim=0),
        )):
            d[f"trf_blocks.{l}.att.heads.{h}.W_query.weight"].copy_(q_h)
            d[f"trf_blocks.{l}.att.heads.{h}.W_key.weight"].copy_(k_h)
            d[f"trf_blocks.{l}.att.heads.{h}.W_value.weight"].copy_(v_h)
        
        if f"trf_blocks.{l}.att.heads.0.W_query.bias" in d:
            q_b, k_b, v_b = torch.chunk(openai_params[f"h.{l}.attn.c_attn.bias"], 3, dim=-1)
            for h, (q_h, k_h, v_h) in enumerate(zip(
                torch.chunk(q_b, n_heads),
                torch.chunk(k_b, n_heads),
                torch.chunk(v_b, n_heads),
            )):
                d[f"trf_blocks.{l}.att.heads.{h}.W_query.bias"].copy_(q_h)
                d[f"trf_blocks.{l}.att.heads.{h}.W_key.bias"].copy_(k_h)
                d[f"trf_blocks.{l}.att.heads.{h}.W_value.bias"].copy_(v_h)
        
        d[f"trf_blocks.{l}.att.out_projection.weight"].copy_(openai_params[f"h.{l}.attn.c_proj.weight"].T)
        d[f"trf_blocks.{l}.att.out_projection.bias"].copy_(openai_params[f"h.{l}.attn.c_proj.bias"])
        
        # Feed-Forward parameters translations
        d[f"trf_blocks.{l}.ff.layers.0.weight"].copy_(openai_params[f"h.{l}.mlp.c_fc.weight"].T)
        d[f"trf_blocks.{l}.ff.layers.0.bias"].copy_(openai_params[f"h.{l}.mlp.c_fc.bias"])
        d[f"trf_blocks.{l}.ff.layers.2.weight"].copy_(openai_params[f"h.{l}.mlp.c_proj.weight"].T)
        d[f"trf_blocks.{l}.ff.layers.2.bias"].copy_(openai_params[f"h.{l}.mlp.c_proj.bias"])
        
        # Layer Normalization layers parameters map
        d[f"trf_blocks.{l}.ln1.scale"].copy_(openai_params[f"h.{l}.ln_1.weight"])
        d[f"trf_blocks.{l}.ln1.shift"].copy_(openai_params[f"h.{l}.ln_1.bias"])
        d[f"trf_blocks.{l}.ln2.scale"].copy_(openai_params[f"h.{l}.ln_2.weight"])
        d[f"trf_blocks.{l}.ln2.shift"].copy_(openai_params[f"h.{l}.ln_2.bias"])
        
    # Final global parameters maps values
    d["final_ln.scale"].copy_(openai_params["ln_f.weight"])
    d["final_ln.shift"].copy_(openai_params["ln_f.bias"])
    d["out_head.weight"].copy_(openai_params["wte.weight"]) # Weight Tying active!


## Step 12: Loading the weights and testing inference

Now we build a **real-sized** GPT and inject the translated weights.

`GPT_CONFIG_REAL` matches the public GPT-2 124M checkpoint exactly. Compare it with the student-sized config from Chapter 3:

| Setting | Chapter 3 student model | This checkpoint |
|---|---|---|
| `vocab_size` | 50,257 | 50,257 |
| `context_length` | 256 | **1,024** (GPT-2's original context window) |
| `emb_dim` | 768 | 768 |
| `n_heads` | 12 | 12 |
| `n_layers` | 12 | 12 |
| `drop_rate` | 0.1 | **0.0** (we are not training, so dropout would only add noise) |
| `qkv_bias` | False | **True** (the original GPT-2 used a bias on Q, K and V) |

The config **must** match the checkpoint. If `qkv_bias` were still `False`, the mapping function would skip the bias copy and the attention math would no longer match GPT-2. If `context_length` were 256, the position-embedding table would be the wrong size for `wpe.weight`.

The test itself is small on purpose:

1. Construct an empty `GPTMODEL` with that config. At this moment the weights are random.
2. `assign_weights(real_gpt, openai_params)` overwrites them with the OpenAI numbers.
3. `real_gpt.eval()` and `torch.no_grad()` put the model in inference mode.
4. We pass the same three token IDs from Chapter 3, `[40, 367, 2885]`, which the GPT-2 tokenizer decodes as the opening of *The Verdict*: **"I HAD"**.

The output shape should be `(1, 3, 50257)`: 1 sequence, 3 positions, 50,257 scores at each position. That is the same shape we got from the untrained model in Chapter 3. The difference is the *meaning* of those scores. They now come from a network that was pretrained on a large slice of the internet, so the highest-scoring next tokens should be ordinary English rather than random vocabulary IDs.

If you want to hear it speak, pass `sample_ids` through `generate_text_simple` from Chapter 3 and decode the result with tiktoken. The architecture, the loss, the training loop, and the pretrained weights are all in place.

In [61]:
GPT_CONFIG_REAL= {
    "vocab_size": 50257, "context_length": 1024, "emb_dim": 768, "n_heads":12, "n_layers":12, "drop_rate":0.0, "qkv_bias":True
}

torch.manual_seed(123)
real_gpt= GPTMODEL(GPT_CONFIG_REAL)
assign_weights(real_gpt, openai_params)
print("structural status: openai pretrained weights successfully mapped and injected")

sample_ids=torch.tensor([[40,367,2885]])
real_gpt.eval()
with torch.no_grad():
    inference_logits= real_gpt(sample_ids)

print("inferende pass result: logits shape is", inference_logits.shape)

structural status: openai pretrained weights successfully mapped and injected
inferende pass result: logits shape is torch.Size([1, 3, 50257])


## Summary

The last chapter gave us a GPT that could run. This chapter gave it a way to learn, a way to measure that learning, and a way to borrow knowledge that already exists.

- **Cross-entropy loss** turns next-token logits into a single number: how surprised the model is by the correct token. Flattening `(batch, tokens, vocab)` down to `(batch * tokens, vocab)` is what lets PyTorch's loss see every position as one classification problem.
- **One training step** is always the same five moves: zero the gradients, forward pass, compute the loss, backward pass, optimizer step. An epoch is that step repeated for every batch in the dataloader.
- **`calc_loss_loader`** scores a dataloader without updating any weights. Putting the model in `eval` mode and wrapping the forward pass in `torch.no_grad()` is how we guard the validation split.
- The **master training loop** stacks epochs, counts tokens seen, and records train versus validation loss so we can tell learning apart from memorising.
- A **checkpoint** is just a `state_dict` on disk. Rebuild the same architecture, load the file, and the model is back.
- **Weight translation** is the bridge between OpenAI's GPT-2 names (`wte`, `c_attn`, `ln_f`, …) and ours (`tok_emb`, per-head `W_query`, `final_ln`, …). Once that mapping is correct, our classes hold the same 124-million-parameter brain as the public checkpoint, including the weight-tied output head.

The architecture can now be trained from scratch, evaluated honestly, saved, reloaded, and initialised from GPT-2. The next step in the series is to **fine-tune**: take this pretrained model and specialise it for a narrower job, such as classification or following instructions.